# Synthetic DiD Demo

Two worked examples from Clarke, Pailañir, Athey & Imbens (2023 IZA):
1. **Proposition 99** — block design, 1 treated unit (California), 38 controls
2. **Gender quotas** — staggered adoption, 9 treated countries, 106 controls

Both datasets are in `references/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sdid import SyntheticDiD

## Example 1: Proposition 99 (California tobacco tax)

In [ ]:
df_prop99 = pd.read_stata('../references/prop99_example.dta')
print(df_prop99.shape)
df_prop99.head()

In [ ]:
model = SyntheticDiD(
    df=df_prop99,
    unit='state',
    time='year',
    outcome='packspercapita',
    treatment='treated',
)

results = model.run(inference='placebo', n_reps=200, seed=1213)
print(results.summary().to_string(index=False))

In [ ]:
# Diagnostic plots
results.plot(show=True)

**IZA paper target:** ATT = -15.604, SE = 9.532, p = 0.102

## Example 2: Gender quotas (staggered adoption)

In [ ]:
df_quota = pd.read_stata('../references/quota_example.dta')

model_quota = SyntheticDiD(
    df=df_quota,
    unit='country',
    time='year',
    outcome='womparl',
    treatment='quota',
)

results_quota = model_quota.run(inference='placebo', n_reps=200, seed=1213)
print(results_quota.summary().to_string(index=False))

**IZA paper target:** ATT = 8.034, SE = 3.740, p = 0.032

## Sensitivity Analysis

In [ ]:
sens = model.sensitivity(
    pre_periods=[5, 10, 20],              # trim pre-treatment window
    drop_outliers=True,                    # drop Z-score outliers
    inference_methods=['placebo', 'bootstrap', 'jackknife'],
    compare_estimators=True,               # DiD vs SC vs SDID
    n_reps=100,                            # fewer reps for speed in demo
    seed=1213,
)

print(sens.table.to_string(index=False))

In [ ]:
# Forest plot of sensitivity results
sens.plot()

## Replication Validation

In [ ]:
from sdid.validate import validate_iza_replication
validate_iza_replication(references_dir='../references/')